# 📘 Deploy llama.cpp Inference Service> **Applicable Environment**: Kubernetes Pod (Ubuntu base image, no Docker/Podman, AMD ROCm environment)> **Purpose**: Compile llama.cpp (HIP/ROCm acceleration), download GGUF models, start and monitor OpenAI-compatible inference service.> **Prerequisites**: First run `1_init_pod_env_cn.md` to complete `/data` symlink; GPU drivers and ROCm tools (`rocminfo` / `amd-smi` / `hipconfig`) are ready.## 1. Service Overview- **Source/Binaries**: `/data/app/llama.cpp` (`bin/llama-server` etc. 120 files, including `libggml-hip.so`)- **Config Scripts**: `/data/service/llamacpp/scripts/` (builder / download / server)- **Model Directory**: `/data/data-store/llamacpp/models/`- **Log Directory**: `/data/data-store/llamacpp/logs/`- **Tuning Documentation**: `/data/service/llamacpp/docs/` (7B / 30B / 80B config and benchmarks)### Local Hardware (Reference docs and actual measurements)| Component | Specs ||------|------|| GPU | AMD Radeon gfx1100 (RDNA3), 96 CU, 48 GB VRAM || CPU | AMD EPYC 9334, 2 Socket, 128 vCPU || RAM | ~503 GB || ROCm | 7.2.1 || llama.cpp | HIP build (`-DGGML_HIP=ON`) |### Default Model: Qwen3-Coder-30B-A3B (Q4_K_M)- Repository: `lucataco/Qwen3-Coder-30B-A3B-Instruct-Q4_K_M-GGUF` (~18.6 GB)- Config: 512K context / 6 concurrent slots / KV `q4_0` / full GPU layers (measured ~90 tok/s, VRAM 32.3/48 GB)---## 2. Clone Source CodeClone llama.cpp source code to persistent directory `/data/app` (survives Pod restarts, no need to re-clone).```python%%bash#!/bin/bashset -euo pipefailSRC="/data/app/llama.cpp"if [ -d "$SRC/.git" ]; then    echo "✅ llama.cpp source already exists: $SRC"    git -C "$SRC" log --oneline -1else    echo "🔧 Cloning llama.cpp ..."    mkdir -p /data/app    git clone https://github.com/ggml-org/llama.cpp.git "$SRC"    echo "✅ Clone complete: $SRC"fi```---## 3. Build (llamacppbuilder.sh)Use `/data/service/llamacpp/scripts/llamacppbuilder.sh` to build; auto-detects GPU (`gfx1100`) and enables HIP acceleration.**Script Highlights**- Default CMake: `-DGGML_HIP=ON -DGGML_HIP_ROCWMMA_FATTN=ON -DGGML_HIP_NO_VMM=ON -DGPU_TARGETS=<auto>`- `--hackathon` mode adds: LTO, HIP graphs, rocWMMA flash attention, native CPU (AVX2/FMA/BMI2), ccache- Output directory: `<source>/build` (default `/data/app/llama.cpp/build`)```python%%bash#!/bin/bashset -euo pipefailBUILDER="/data/service/llamacpp/scripts/llamacppbuilder.sh"# 0) View parameters and hardware# bash "$BUILDER" --help# 1) Auto-detect GPU architecture (should be gfx1100 on this machine)detect_gpu() {    rocminfo 2>/dev/null | grep -m1 'Name:.*gfx' | awk '{print $2}' || echo "gfx1100"}echo "GPU target: $(detect_gpu)"# 2) Build (hackathon mode: maximum optimization, full-core parallelism)# Estimated 10~30 minutes, please wait patiently# bash "$BUILDER" --hackathon```**After build completes**, move `build/bin/` to source root for unified management (already done in current environment):```python%%bash#!/bin/bashset -euo pipefailcd /data/app/llama.cppif [ -d "build/bin" ]; then    echo "🔧 Moving build/bin -> ./bin ..."    mv build/bin/ .fils bin/ | grep -E "llama-server|llama-cli|llama-embedding"```> 💡 If binaries already exist, you can use directly: `ls /data/app/llama.cpp/bin/llama-server`---## 4. Install Model Download Tool`download_model.sh` depends on HuggingFace CLI `hf` (huggingface_hub). For domestic networks, it is recommended to use mirror `https://hf-mirror.com` (enabled by default in script).```python%%bash#!/bin/bashset -euo pipefailif ! command -v hf &> /dev/null; then    echo "🔧 Installing huggingface_hub ..."    pip install --break-system-packages -q huggingface_hub    echo "✅ Install complete"fihf --version```---## 5. Download Model (download_model.sh)```python%%bash#!/bin/bashset -euo pipefail# 30B Q4_K_M (~18.6GB, current deployment) — domestic mirror# bash /data/service/llamacpp/scripts/download_model_30b.sh 30b# 80B (~47GB, Q4_K_M)# bash /data/service/llamacpp/scripts/download_model.sh 80b# Force re-download / official source# bash /data/service/llamacpp/scripts/download_model_30b.sh 30b --force --no-mirror# View helpbash /data/service/llamacpp/scripts/download_model_30b.sh --help```> Note: For 30B Q4_K_M use `download_model_30b.sh`; `download_model.sh`'s `30b` is Q8_0 version.After download, models are located at:| Model | Path ||------|------|| 30B Q4_K_M | `/data/data-store/llamacpp/models/Qwen3-Coder-30B-A3B-Q4_K_M/qwen3-coder-30b-a3b-instruct-q4_k_m.gguf` || 80B Q4_K_M | `/data/data-store/llamacpp/models/Qwen3-Coder-Next-Opus-Distilled-Q4_K_M/*.gguf` |> ✅ Unified: `llamaserver.sh` default `BASE_MODEL_DIR=/data/data-store/llamacpp/models`, consistent with download script directory, no extra settings needed.---## 6. Start Service (llamaserver.sh)Unified management script: `start / stop / status / test`, supports `--model 30b`, `--port`, `--verbose`, `--dry-run`.**Startup parameters (30B)**: `-ngl -1` (full GPU), `-c 524288` (512K), `-np 6`, KV `q4_0`, `--flash-attn on`, `--jinja`, `--numa distribute`.```python%%bash#!/bin/bashset -euo pipefailcd /data/service/llamacpp/scripts# Start 30B (default model directory already consistent with download script)bash llamaserver.sh start --model 30b# Specify port# bash llamaserver.sh start --port 8081# Preview only the command to be executed# bash llamaserver.sh start --dry-run# Check statusbash llamaserver.sh status```**Verify readiness**:```python%%bash#!/bin/bashset -euo pipefail# Health check (returns "ok" when ready)curl -s --max-time 5 http://localhost:8080/health; echo```---## 7. Inference Testing```python%%bash#!/bin/bashset -euo pipefail# One-click test (health check + OpenAI-compatible inference + speed/token stats)bash /data/service/llamacpp/scripts/llamaserver.sh test# Or directly call OpenAI-compatible endpointcurl -s --max-time 60 http://localhost:8080/v1/chat/completions \    -H "Content-Type: application/json" \    -d '{"model":"test","messages":[{"role":"user","content":"Please introduce yourself in one sentence"}],"max_tokens":50,"temperature":0.7}' \    | python3 -m json.tool --no-ensure-ascii```**Common Endpoints**| Endpoint | Description ||------|------|| `GET /health` | Health check || `GET /v1/models` | Model list || `POST /v1/chat/completions` | Chat completion || `POST /v1/embeddings` | Embedding (requires `--embedding`) || `GET /props` | Runtime parameter details |---## 8. Monitoring (monitor.sh)`/data/service/monitor.sh`: Text-based monitoring panel refreshed every 10 seconds, showing llama.cpp (8080), unires backend (8000), frontend (7860) status, and CPU / memory / GPU utilization and VRAM.```python%%bash#!/bin/bashset -euo pipefail# Run monitoring panel (exit with Ctrl+C)# bash /data/service/monitor.sh# Manual quick viewecho "== llama-server process =="pgrep -a llama-server || echo "Not running"echoecho "== GPU VRAM =="amd-smi metric --mem 2>/dev/null | grep -E "TOTAL_VRAM|USED_VRAM" | head -4 || rocm-smi --showmeminfo vram 2>/dev/null```---## 9. Service Management and Logs```python%%bash#!/bin/bashset -euo pipefail# Stop (graceful SIGTERM, force kill on timeout; double fallback by PID file + port)# bash /data/service/llamacpp/scripts/llamaserver.sh stop# LogsLOG_DIR=/data/data-store/llamacpp/logsls -la "$LOG_DIR" 2>/dev/null || echo "（No logs yet, service has never been started）"```**Common Operations Commands**```bashtail -f /data/data-store/llamacpp/logs/llamaserver-qwen3.log   # Tail logsss -tlnp | grep 8080                                          # Port occupationrocm-smi --showmeminfo vram                                   # VRAM usage```**One-click Pod restart recovery** (similar to PostgreSQL's `init-pg.sh`):```bash/data/init/init-llamacpp.sh   # Install dependencies → compile if binaries missing → download if model missing → start (wait up to 10 minutes)```---## 10. Troubleshooting| Issue | Solution ||------|------|| `hf` command not found | `pip install --break-system-packages huggingface_hub` || Download network unreachable | Script defaults to `HF_ENDPOINT=https://hf-mirror.com`, or use `--no-mirror` || Model not found | Check `BASE_MODEL_DIR` matches download directory (see Section 5) || Startup timeout/failure | `cat /data/data-store/llamacpp/logs/llamaserver-*.log` || First startup very slow | Model is on PVC network storage, 18.6GB read/upload to VRAM takes 5~10 minutes, normal; `status` shows process alive, just wait || VRAM insufficient | Reduce context `-c`, concurrency `-np`, or KV quantization (`q4_0`) || Segmentation fault/illegal instruction | Confirm binary is HIP build and `GPU_TARGETS` is correct (`gfx1100`) || Build slow | Use `--hackathon` or reduce `-j`; first run `cmake --build build -j 128` |For tuning details, see `/data/service/llamacpp/docs/01_7B_model_config.md` ~ `05_tuning_history.md` and `scripts/llama-server-params.md`.---## 11. Next Steps- Embedding: For RAG scenarios, use `--embedding` (or refer to document #3 for pgvector vector storage).- Performance benchmark: Use `llamaserver.sh test` to record tok/s; compare against `docs/04_30B_tuning_benchmark.md`.- Persistence: Models are already on PVC (`/data/data-store`), no re-download needed after Pod restart, just run `llamaserver.sh start`.- Monitoring: Use `/data/service/monitor.sh` to observe PostgreSQL, unires backend/frontend uniformly.---> ✅ At this point, llama.cpp inference service deployment is complete; can be integrated into Uni-Resource Agent application (refer to `2_app_cn.md`).